# Soup Kernel Challenge: faster streamed training on T4

Settings: **GPU T4 x2**, **Internet on**. This notebook tests the candidate,
runs base/solution/solution/base, checks all 10 rules and scores the result.
Commands fail immediately on errors. GPU correctness and performance are pending until this executes.

The candidate removes repeated head GEMMs, skips head padding, keeps full-shape MLP on T4,
uses a fused NF4 decoder, and retains decoder activations on GPU for the fixed task.
The model weights remain resident on the GPU; **599 MiB is not an expected or promised result**.
The speed target is <=40s total; this candidate is unmeasured. Compare all 10 rules, not only speed.
Larger/unsupported calls retain CPU offload. Cold compilation overlaps Trainer imports and model load.

Push the local changes to GitHub before running: Kaggle clones that repository.
The repository and development branch are filled in below. For a reproducible final
submission, replace COMMIT with the full tested SHA, Save & Run All, publish the saved version,
and put its link in both README.md and SUBMISSION.md. Do not run other GPU jobs.


In [ ]:
REPO_URL = "https://github.com/Ser1kbol077/soup-kernel-project.git"
COMMIT = "codex/t4-mlp-cast-elision"  # development branch; replace with the full immutable SHA for the final benchmark


In [ ]:
import importlib.metadata as metadata
import os
import subprocess
import sys

gpu_names = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True
)
print(gpu_names)
assert gpu_names.strip() and all("T4" in name for name in gpu_names.splitlines()), (
    "Select Kaggle GPU T4 x2"
)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

requirements = [
    "soup-cli[train]==0.77.0", "transformers==5.16.1", "peft==0.20.0",
    "trl==0.29.1", "bitsandbytes==0.50.2", "accelerate==1.14.0", "pytest",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *requirements], check=True)
for package in ["soup-cli", "torch", "triton", "transformers", "peft", "trl",
                "bitsandbytes", "accelerate"]:
    print(package, metadata.version(package))


In [ ]:
import base64
import tempfile
from pathlib import Path

token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass
auth = []
if token:
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    auth = ["-c", f"http.extraHeader=Authorization: Basic {basic}"]

# A new empty directory on every run; no previous results are deleted.
REPO = Path(tempfile.mkdtemp(prefix="soup-vocab-", dir="/kaggle/working"))
clone = subprocess.run(
    ["git", *auth, "clone", "-q", REPO_URL, str(REPO)], capture_output=True, text=True
)
if clone.returncode:
    raise RuntimeError("git clone failed: " + clone.stderr)
if COMMIT:
    fetched = subprocess.run(
        ["git", *auth, "-C", str(REPO), "fetch", "-q", "origin", COMMIT],
        capture_output=True, text=True,
    )
    if fetched.returncode:
        raise RuntimeError("git fetch failed: " + fetched.stderr)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "-q", "FETCH_HEAD"],
                   check=True)
auth = []
token = basic = ""
TESTED_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
print("Repository:", REPO)
print("Tested commit:", TESTED_COMMIT)
# Fail before testing if Kaggle fetched an older candidate without the memory changes.
import ast
CANDIDATE_FILES = (
    "solution/streamed_head.py",
    "solution/streamed_head_patch.py",
    "solution/activation_offload.py",
    "solution/mlp_padding.py",
    "solution/nf4_decode.py",
    "solution/compile_overlap.py",
)
missing = [name for name in CANDIDATE_FILES if not (REPO / name).is_file()]
assert not missing, f"Push the new candidate to GitHub or select its SHA; missing: {missing}"
init_tree = ast.parse((REPO / "solution/__init__.py").read_text(encoding="utf-8"))
wired = {
    node.module for node in ast.walk(init_tree)
    if isinstance(node, ast.ImportFrom) and node.level == 1
}
assert {"streamed_head_patch", "activation_offload", "mlp_padding", "nf4_decode"} <= wired, (
    "The cloned solution.apply() does not include the new memory patches"
)
print("Memory candidate source files:", *CANDIDATE_FILES, sep="\n  ")
GPU_ENV = dict(os.environ, CUDA_VISIBLE_DEVICES="0", CUDA_DEVICE_ORDER="PCI_BUS_ID")

def run(*arguments, env=GPU_ENV):
    subprocess.run([sys.executable, *arguments], cwd=REPO, env=env, check=True)


In [ ]:
run("task/prepare_data.py")
# Verify the installed package contracts before allocating the test model.
run("-c", "\n".join([
    "import solution; solution.apply()",
    "from solution.compile_overlap import join_compile_overlap; join_compile_overlap()",
    "from solution.streamed_head import CHUNK_ROWS",
    "assert CHUNK_ROWS == 256, 'old head implementation: push the speed candidate'",
    "from bitsandbytes import functional as bnb_functional",
    "assert getattr(bnb_functional.dequantize_4bit, '_soup_nf4_decode_patch', False), 'NF4 patch missing'",
    "from trl.trainer.sft_trainer import SFTTrainer",
    "from transformers.models.qwen2.modeling_qwen2 import Qwen2ForCausalLM, Qwen2DecoderLayer, Qwen2MLP",
    "assert getattr(SFTTrainer.compute_loss, '_soup_streamed_head_patch', False), 'streamed trainer not installed'",
    "assert getattr(Qwen2ForCausalLM.forward, '_soup_streamed_head_patch', False), 'streamed head not installed'",
    "assert getattr(Qwen2DecoderLayer.forward, '_soup_activation_offload_patch', False), 'activation offload not installed'",
    "assert getattr(Qwen2MLP.forward, '_soup_mlp_padding_patch', False), 'MLP padding patch missing'",
    "print('Speed candidate installed: head VJP, SM75 MLP fallback, resident decoder, NF4 decode')",
]))
# Compilation and all CUDA numerical/hook tests must complete on the T4.
run("-m", "pytest", "solution", "-q")
run("-m", "pytest", "tests", "-q",
    env=dict(GPU_ENV, SOUP_APPLY_SOLUTION="1"))
# Test processes have exited; the notebook itself has not initialized CUDA.


In [ ]:
# Cold compiler caches and measurement environment are managed by the unmodified harness.
run("harness/run_task.py", "--repeats", "2", "--profile")


In [ ]:
# Print the measured GPU peaks even if a subsequent acceptance check fails.
import json
result = json.loads((REPO / "runs/result.json").read_text(encoding="utf-8"))
for side in ("base", "solution"):
    for row in result["runs"][side]:
        m = row["metrics"]
        print(
            row["run"],
            "full:", row["wall_s"], "s;",
            "training loop:", m.get("train_loop_s"), "s;",
            "outside loop:", round(row["wall_s"] - m["train_loop_s"], 3) if m.get("train_loop_s") else None, "s;",
            "apply:", m.get("apply_s"), "s;",
            "external peak:", row["gpu_peak_mib"], "MiB;",
            "torch allocated:", m.get("torch_max_allocated_mib"), "MiB;",
            "torch reserved:", m.get("torch_max_reserved_mib"), "MiB",
        )
profile_file = REPO / "runs/solution_profile/profile.json"
if profile_file.exists():
    profile = json.loads(profile_file.read_text(encoding="utf-8"))
    print("Largest profiled GPU kernels (separate diagnostic run):")
    for kernel in profile.get("kernels", [])[:20]:
        print(round(kernel["time_us"] / 1000, 3), "ms;", kernel["count"], "calls;", kernel["name"])
summary = result["summary"]
print(json.dumps(summary, indent=2))
print("Resident decoder trades GPU memory for fewer PCIe copies; verify the measured peak.")

# Preserve diagnostics even when a rule fails; the checker exit status is retained.
try:
    run("harness/check_rules.py")
finally:
    import zipfile
    archive = REPO.parent / f"{REPO.name}-{TESTED_COMMIT[:12]}-logs.zip"
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
        bundle.writestr("tested_commit.txt", TESTED_COMMIT + "\n")
        for path in sorted((REPO / "runs").rglob("*")):
            if path.is_file() and path.suffix in (".json", ".log"):
                bundle.write(path, str(path.relative_to(REPO)))
    print("Diagnostic archive:", archive)
run("harness/score.py", "runs/result.json")


In [ ]:
import json

result = json.loads((REPO / "runs/result.json").read_text())
checks = json.loads((REPO / "runs/checks.json").read_text())
assert len(checks["checks"]) == 10 and all(row["ok"] for row in checks["checks"])
# Use the harness's own source classifier; the shipped example is excluded.
sys.path.insert(0, str(REPO / "harness"))
from check_rules import source_kernels
own, *_ = source_kernels(REPO / "solution")
solution_runs = result["runs"]["solution"]
common = set(own)
for row in solution_runs:
    metrics = row["metrics"]
    active = {
        name for name, count in metrics["triton_launches"].items()
        if count > 0 and metrics["triton_files"].get(name, "").replace("\\", "/").startswith("solution/")
    }
    common &= active
for row in solution_runs:
    metrics = row["metrics"]
    own_ms = sum(metrics["own_kernel_ms"].get(name, 0) for name in common)
    share = own_ms / (1000 * metrics["timed_loop_s"])
    print(row["run"], "original-kernel share:", f"{share:.2%}")
print(json.dumps(result["summary"], indent=2))
print("Tested commit:", TESTED_COMMIT)
print("Artifacts:", REPO / "runs/result.json", REPO / "runs/checks.json")


Optional profiling **after** the official measurements:

`run("harness/run_task.py", "--only", "solution", "--profile", "--out", "runs_profile")`

The solution-only run cannot satisfy checks requiring a baseline. If own-kernel share
is below 5%, this version is not yet admissible: inspect the useful operations in the
profile and implement the next optimization. Do not add artificial work or delay.

Download the diagnostic logs ZIP, runs/result.json and runs/checks.json.
Compare both solution repeats; use the largest GPU peak, as the harness does. A test skip or a static source
scan does not establish T4 correctness. Fill the submission form only from real runs.
